## Phase 2 — Harris Corner Detection
## Detecting stable feature points inside a user-defined bounding box
**Ali | 23L-2619 | BS Data Science 6th Semester | Fundamentals of Computer Vision | Final Project**

In [1]:
import sys, os, cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

PROJECT_ROOT = r"C:\Users\pakistan\OneDrive\Desktop\cv_project"
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

for mod in list(sys.modules.keys()):
    if any(x in mod for x in ["core","utils","metrics","visualization"]):
        del sys.modules[mod]

from utils.config_loader import ConfigLoader
from core.harris_detector import HarrisDetector

cfg     = ConfigLoader("config.yaml").config
harris  = HarrisDetector(cfg)

print("✅ Setup complete")

✅ Setup complete


In [2]:
import glob
import os
import sys

from metrics.logger import AppLogger
AppLogger.reset()

pyc_files = glob.glob("../metrics/__pycache__/*.pyc")
for pyc in pyc_files:
    if os.path.exists(pyc):
        os.remove(pyc)

for mod in list(sys.modules.keys()):
    if any(x in mod for x in ["core", "utils", "metrics", "visualization"]):
        del sys.modules[mod]

In [3]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
matplotlib.rcParams['figure.dpi'] = 100

cap = cv2.VideoCapture(cfg.input.source)
ret, frame = cap.read()
cap.release()

if not ret:
    raise RuntimeError("Could not read video frame.")

# Downsample for display only
display_frame = cv2.resize(frame, (1280, 720), interpolation=cv2.INTER_LINEAR)

gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
rgb  = cv2.cvtColor(display_frame, cv2.COLOR_BGR2RGB)

print(f"Frame loaded: {frame.shape[1]}x{frame.shape[0]}")

fig, ax = plt.subplots(figsize=(12, 6))
ax.imshow(rgb)
ax.set_title("First Frame", fontsize=13)
ax.axis("off")
plt.tight_layout()

os.makedirs(r"outputs\phase_2", exist_ok=True)
save_path = r"outputs\phase_2\first_frame.png"
fig.savefig(save_path, bbox_inches="tight", dpi=100)
plt.close(fig)

print(f"Image saved to: {save_path}")

Frame loaded: 3840x2160
Image saved to: outputs\phase_2\first_frame.png


In [4]:
# ── Step 1: Scrub through video to find a frame with the object ──────
DISP_W, DISP_H = 1280, 720

cap_scan = cv2.VideoCapture(cfg.input.source)
total_frames = int(cap_scan.get(cv2.CAP_PROP_FRAME_COUNT))

def nothing(x): pass

cv2.namedWindow("Scrub to find object — SPACE to confirm", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Scrub to find object — SPACE to confirm", DISP_W, DISP_H)
cv2.createTrackbar("Frame", "Scrub to find object — SPACE to confirm",
                   0, total_frames - 1, nothing)

chosen_frame = None
chosen_frame_idx = 0

while True:
    pos = cv2.getTrackbarPos("Frame", "Scrub to find object — SPACE to confirm")
    cap_scan.set(cv2.CAP_PROP_POS_FRAMES, pos)
    ret, f = cap_scan.read()
    if not ret:
        continue

    disp = cv2.resize(f, (DISP_W, DISP_H), interpolation=cv2.INTER_LINEAR)
    cv2.putText(disp, f"Frame {pos}/{total_frames-1}  |  SPACE = select this frame  |  ESC = cancel",
                (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
    cv2.imshow("Scrub to find object — SPACE to confirm", disp)

    key = cv2.waitKey(30) & 0xFF
    if key == 32:  # SPACE — confirm this frame
        chosen_frame = f
        chosen_frame_idx = pos
        print(f"✅ Frame {pos} selected.")
        break
    elif key == 27:  # ESC — cancel
        print("Cancelled.")
        break

cap_scan.release()
cv2.destroyAllWindows()

if chosen_frame is None:
    raise RuntimeError("No frame selected. Re-run this cell.")

✅ Frame 0 selected.


In [5]:
# ── Step 2: Draw bbox on the chosen frame ────────────────────────────
orig_h, orig_w = chosen_frame.shape[:2]
scale_x = orig_w / DISP_W
scale_y = orig_h / DISP_H

disp_chosen = cv2.resize(chosen_frame, (DISP_W, DISP_H), interpolation=cv2.INTER_LINEAR)
hint = disp_chosen.copy()
cv2.putText(hint, "Draw box around object — press ENTER to confirm",
            (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

bbox_disp = cv2.selectROI("Select Object", hint, fromCenter=False, showCrosshair=True)
cv2.destroyAllWindows()

if bbox_disp == (0, 0, 0, 0):
    raise ValueError("No region selected. Re-run this cell.")

# Scale bbox back to full resolution
x = int(bbox_disp[0] * scale_x)
y = int(bbox_disp[1] * scale_y)
w = int(bbox_disp[2] * scale_x)
h = int(bbox_disp[3] * scale_y)
bbox = (x, y, w, h)

# Full-res gray of chosen frame (for Harris + LK)
frame  = chosen_frame          # reassign so rest of notebook uses this frame
gray   = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

# Phase 3 also needs gray1/frame1 for LK
frame1 = chosen_frame
gray1  = gray

print(f"Chosen frame index : {chosen_frame_idx}")
print(f"Bbox (display res) : {bbox_disp}")
print(f"Bbox (full res)    : x={x}, y={y}, w={w}, h={h}")

mask   = harris.create_bbox_mask(gray.shape, bbox)
points = harris.detect(gray, mask=mask)
print(f"Harris corners detected: {len(points)}")

Chosen frame index : 0
Bbox (display res) : (704, 576, 72, 102)
Bbox (full res)    : x=2112, y=1728, w=216, h=306
Harris corners detected: 201


In [6]:
vis = cv2.cvtColor(frame.copy(), cv2.COLOR_BGR2RGB)

# Draw bounding box
cv2.rectangle(vis, (x,y), (x+w, y+h), (0,220,0), 2)

# Draw Harris corners
for pt in points:
    cx, cy = int(pt[0,0]), int(pt[0,1])
    cv2.circle(vis, (cx,cy), 4, (255,255,0), -1)

plt.figure(figsize=(12,6))
plt.imshow(vis)
plt.title(f"Harris Corners Detected: {len(points)} points (yellow dots)", fontsize=13)
plt.axis("off")
plt.tight_layout()
os.makedirs(r"outputs\phase_2", exist_ok=True)
save_path = r"outputs\phase_2\harris_corners.png"
plt.savefig(save_path, bbox_inches="tight", dpi=100)
plt.close()
print(f"Image saved to: {save_path}")

Image saved to: outputs\phase_2\harris_corners.png


In [7]:
# Show raw Harris response heatmap inside ROI
roi_gray = gray[y:y+h, x:x+w].astype(np.float32)
response = cv2.cornerHarris(roi_gray, blockSize=cfg.harris.block_size,
                             ksize=3, k=cfg.harris.k)
response = cv2.dilate(response, None)  # enhance for visibility

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].imshow(cv2.cvtColor(frame[y:y+h, x:x+w], cv2.COLOR_BGR2RGB))
axes[0].set_title("ROI — Original", fontsize=12)
axes[0].axis("off")

axes[1].imshow(response, cmap="hot")
axes[1].set_title("Harris Response Map (brighter = stronger corner)", fontsize=12)
axes[1].axis("off")

plt.suptitle("Harris Corner Detection — Response Visualization", fontsize=14, fontweight="bold")
plt.tight_layout()

os.makedirs(r"outputs\phase_2", exist_ok=True)
save_path = r"outputs\phase_2\harris_response.png"
fig.savefig(save_path, bbox_inches="tight", dpi=100)
plt.close(fig)
print(f"Plot saved to: {save_path}")

print(f"Max response  : {response.max():.4f}")
print(f"Threshold used: {cfg.harris.quality_level * response.max():.4f}")

Plot saved to: outputs\phase_2\harris_response.png
Max response  : 29195798.0000
Threshold used: 145978.9900


In [8]:
# Show how quality_level affects number of corners detected
quality_levels = [0.001, 0.005, 0.01, 0.05, 0.1]
counts = []

fig, axes = plt.subplots(1, len(quality_levels), figsize=(16, 4))

for i, ql in enumerate(quality_levels):
    # Temporarily patch config
    cfg.harris.quality_level = ql
    pts = harris.detect(gray, mask=mask)
    counts.append(len(pts))

    vis2 = cv2.cvtColor(frame[y:y+h, x:x+w].copy(), cv2.COLOR_BGR2RGB)
    for pt in pts:
        px, py = int(pt[0,0]) - x, int(pt[0,1]) - y
        if 0 <= px < w and 0 <= py < h:
            cv2.circle(vis2, (px,py), 3, (255,255,0), -1)

    axes[i].imshow(vis2)
    axes[i].set_title(f"ql={ql}\n{len(pts)} pts", fontsize=10)
    axes[i].axis("off")

# Restore original
cfg.harris.quality_level = ConfigLoader("config.yaml").config.harris.quality_level

plt.suptitle("Effect of quality_level on Harris Corner Count", fontsize=13, fontweight="bold")
plt.tight_layout()

os.makedirs(r"outputs\phase_2", exist_ok=True)
save_path = r"outputs\phase_2\quality_level_comparison.png"
fig.savefig(save_path, bbox_inches="tight", dpi=100)
plt.close(fig)
print(f"Plot saved to: {save_path}")

for ql, c in zip(quality_levels, counts):
    print(f"quality_level={ql:.3f}  →  {c} corners")

Plot saved to: outputs\phase_2\quality_level_comparison.png
quality_level=0.001  →  201 corners
quality_level=0.005  →  201 corners
quality_level=0.010  →  201 corners
quality_level=0.050  →  201 corners
quality_level=0.100  →  201 corners


In [9]:
print(f"Start frame: {chosen_frame_idx}")
print(f"bbox: {bbox}")

Start frame: 0
bbox: (2112, 1728, 216, 306)


In [27]:
## Summary
#- Harris corner detector identifies stable, texture-rich points inside the user bbox
#- The response map shows where corners are strongest (bright = strong corner)
#- `quality_level` controls the threshold — lower = more corners, higher = fewer but stronger
#- These corners become the tracking anchors for the LK optical flow in Phase 3 ✅